# 03 · Baselines and ranking metrics

**Learning objectives**

1. Compute precision, recall, hit-rate, MAP and NDCG **by hand**, and know which question each one answers.
2. Build the three baselines every recommender project needs: **random**, **popularity** and **trending**.
3. Understand why popularity is hard to beat and what beyond-accuracy metrics (coverage, novelty) reveal.
4. See how the **definition of relevance** changes the leaderboard.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from recsys.data import load_or_synthesize
from recsys.pipeline import prepare_data
from recsys import metrics as M
from recsys.models import RandomRecommender, PopularityRecommender, TrendingRecommender
from recsys.plotting import set_style, bar_leaderboard, model_color

set_style()
events, item_categories, SOURCE = load_or_synthesize()
data = prepare_data(events)

## 1. Metrics by hand

One user. We recommended 5 items, and 3 items turned out to be relevant (one of them we never recommended):

```
rank:        1    2    3    4    5
recommended: A    B    C    D    E
relevant?    ✗    ✓    ✗    ✗    ✓        relevant set = {B, E, Z}
```

Work these out on paper first, then check them below.

| metric | formula | value |
|---|---|---|
| precision@5 | hits / k | 2/5 = 0.40 |
| recall@5 | hits / \|relevant\| | 2/3 = 0.67 |
| hit-rate@5 | any hit? | 1 |
| MRR | 1 / rank of first hit | 1/2 = 0.50 |
| AP@5 | mean of precision *at each hit*, over min(\|rel\|, k) | (1/2 + 2/5) / 3 = 0.30 |
| NDCG@5 | DCG / ideal DCG, with $\text{DCG} = \sum_r \frac{\text{hit}_r}{\log_2(r+1)}$ | see below |

NDCG rewards putting hits **early**. The ideal list for this user would put 3 hits at ranks 1, 2 and 3:

$$\text{DCG} = \frac{1}{\log_2 3} + \frac{1}{\log_2 6} \approx 1.017, \qquad \text{IDCG} = 1 + \frac{1}{\log_2 3} + \frac{1}{\log_2 4} \approx 2.131, \qquad \text{NDCG} \approx 0.477$$

In [ ]:
recs, relevant = np.array(list("ABCDE")), np.array(list("BEZ"))
h = M.hits_vector(recs, relevant)
pd.Series({
    "hits": h.tolist(),
    "precision@5": M.precision_at_k(h, 5),
    "recall@5": M.recall_at_k(h, 3, 5),
    "hit_rate@5": M.hit_rate_at_k(h, 5),
    "MRR": M.reciprocal_rank(h),
    "AP@5": M.average_precision_at_k(h, 3, 5),
    "NDCG@5": M.ndcg_at_k(h, 3, 5),
})

**Which one should you report?**

* **NDCG@10** is the primary metric here: it is position-aware and comparable across users.
* **Recall@10** is good for candidate generators ("did the right items make the shortlist?").
* **Hit-rate@10** is easiest to explain to stakeholders ("x% of users got at least one useful recommendation").
* Report a few metrics and look for *agreement*. If they disagree, find out why before declaring a winner.

## 2. Baselines

* **Random**: the floor. If a model cannot beat it, there is a bug.
* **Popularity**: the items the most distinct users interacted with. Everyone gets the same list.
* **Trending**: popularity with exponential time decay. Last week counts more than last month.

In [ ]:
fit_kw = dict(train_events=data.train_events, item_map=data.item_map)
baselines = {
    "random": RandomRecommender().fit(data.X_train),
    "popularity": PopularityRecommender().fit(data.X_train),
    "trending": TrendingRecommender(half_life_days=7).fit(data.X_train, **fit_kw),
}
board = M.evaluate_many(baselines, data.X_train, data.ground_truth)
board.round(4)

In [ ]:
bar_leaderboard(board, "ndcg@10", title="Baselines: NDCG@10")
plt.show()

Look at three things:

1. **Popularity crushes random.** Behaviour is concentrated on few items (notebook 01), so "recommend what everyone likes" is a strong guess.
2. **Trending vs popularity.** If the catalogue changes quickly (new products, seasonal demand), recency helps.
3. **Coverage and novelty.** Popularity recommends a tiny slice of the catalogue, the same items to everyone. Random has perfect coverage and is useless. Good personalised models should sit in between.

> **Principal DS note.** In production, the honest comparison is your model **vs the best non-personalised baseline**, never vs random. Many published "improvements" disappear against a well-tuned popularity baseline.

## 3. How metrics change with k

In [ ]:
ks = [1, 3, 5, 10, 20, 50]
fig, ax = plt.subplots()
for name, model in baselines.items():
    vals = [M.evaluate(model, data.X_train, data.ground_truth, k=k)[f"recall@{k}"] for k in ks]
    ax.plot(ks, vals, marker="o", label=name, color=model_color(name))
    ax.annotate(name, (ks[-1], vals[-1]), xytext=(4, 0), textcoords="offset points", va="center", fontsize=9)
ax.set_xscale("log"); ax.set_xticks(ks); ax.set_xticklabels(ks)
ax.set_xlabel("k"); ax.set_ylabel("recall@k"); ax.set_title("Recall grows with list length")
ax.legend(loc="upper left")
plt.show()

## 4. What counts as "relevant"?

So far any test-week interaction counts, even a single view. The business cares about **commercial intent**. Let's re-run with only add-to-carts and purchases as relevant.

In [ ]:
data_buy = prepare_data(events, target_events=("addtocart", "transaction"))
print("evaluation users:", len(data_buy.ground_truth), "(vs", len(data.ground_truth), "for any event)")
fit_kw_buy = dict(train_events=data_buy.train_events, item_map=data_buy.item_map)
board_buy = M.evaluate_many({
    "random": RandomRecommender().fit(data_buy.X_train),
    "popularity": PopularityRecommender().fit(data_buy.X_train),
    "trending": TrendingRecommender().fit(data_buy.X_train, **fit_kw_buy),
}, data_buy.X_train, data_buy.ground_truth)
board_buy.round(4)

There are far fewer evaluation users, so expect **noisier** estimates (notebook 07 adds confidence intervals). Choose the relevance definition that matches the business goal, and say which one you used every time you report a number.

## Key takeaways

* Know what each metric rewards. NDCG cares about position, recall cares about the shortlist.
* Popularity is a strong baseline on long-tailed data, and it is the one to beat.
* Accuracy alone hides **coverage** and **novelty** problems.
* The relevance definition is a product decision. Report it.

## Exercises

1. **(Easy)** Compute NDCG@5 by hand for `recommended = [B, E, A, C, D]` with the same relevant set. Check your answer with `M.ndcg_at_k`.
2. **(Medium)** Implement *category popularity*: recommend the most popular items from the user's most-viewed category. Does it beat global popularity? (Needs `item_categories`.)
3. **(Medium)** Sweep `half_life_days` in `TrendingRecommender` over `[1, 3, 7, 14, 30]` and plot NDCG@10. What does the best value tell you about the dataset?
4. **(Stretch)** Popularity is the same list for everyone, so the test users who interact with popular items get "free" hits. Split NDCG by whether the user's relevant items are mostly head or tail items.